# KRIOS GIS - ArcGIS Layer Check

This notebook keeps only the required target-layer check for:
- Kytkinlaitokset_Fingrid / Sähköasemat_liityntäkapasiteetti

## Native ArcGIS API Workflow

This notebook now uses two code chunks:

1. Chunk 1: list all FeatureServer services and related info from the native ArcGIS services API
2. Chunk 2: extract the required FeatureServer layer and show its attribute values

In [3]:
import pandas as pd
import requests

BASE_URL = "https://services2.arcgis.com/uh3cDCipmuPcmxmx/ArcGIS/rest/services"

# Chunk 1: list all native ArcGIS services and keep FeatureServer items.
root_resp = requests.get(f"{BASE_URL}?f=pjson", timeout=45)
print("Root status:", root_resp.status_code)
root_resp.raise_for_status()
root_json = root_resp.json()

services = root_json.get("services", [])
services_df = pd.DataFrame(services)

if services_df.empty:
    raise RuntimeError("No services returned from ArcGIS root endpoint.")

services_df["service_url"] = (
    BASE_URL + "/" + services_df["name"].astype(str) + "/" + services_df["type"].astype(str)
    )
feature_services_df = services_df[services_df["type"].eq("FeatureServer")].copy()

print("Total services:", len(services_df))
print("FeatureServer count:", len(feature_services_df))
display(feature_services_df[["name", "type", "service_url"]].sort_values("name").reset_index(drop=True))

Root status: 200
Total services: 113
FeatureServer count: 113


,name,type,service_url
0,Asemakaavoitettualue_VESISTOALUEET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
1,Asuinrakennus,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
2,Biokaasu,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
3,EVO_OYK,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
4,EVO_OYK_KIINTEISTÖTUNNUKSET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
...,...,...,...
108,survey123_af37f13c356e4d2581a14b302ed42ec9_res...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
109,survey123_d8fbac0dec194234a9b66a751e1c8b8a_form,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
110,survey123_f9551175b74545e8a56f137d280efef9_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
111,survey123_fa3c5b79a00e48b88cf0e75cbd493ac0_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...


In [ ]:
# Chunk 2: extract required FeatureServer layer and show attribute values.
TARGET_SERVICE_NAME = "Kytkinlaitokset_Fingrid"
TARGET_LAYER_HINT = "liityntäkapasiteetti"

target_service = feature_services_df[feature_services_df["name"] == TARGET_SERVICE_NAME].copy()
if target_service.empty:
    raise RuntimeError(f"Target service not found: {TARGET_SERVICE_NAME}")

service_url = target_service.iloc[0]["service_url"]
service_resp = requests.get(f"{service_url}?f=pjson", timeout=45)
print("Service status:", service_resp.status_code)
service_resp.raise_for_status()
service_json = service_resp.json()

layers = service_json.get("layers", [])
layers_df = pd.DataFrame(layers)
if layers_df.empty:
    raise RuntimeError("No layers found inside target FeatureServer.")

layers_df["name_lc"] = layers_df["name"].astype(str).str.lower()
layer_match = layers_df[layers_df["name_lc"].str.contains(TARGET_LAYER_HINT.lower(), na=False)].copy()
selected_layer = layer_match.iloc[0] if not layer_match.empty else layers_df.iloc[0]

layer_url = f"{service_url}/{int(selected_layer['id'])}"
print("Selected layer:", selected_layer["name"])
print("Layer URL:", layer_url)

meta_resp = requests.get(f"{layer_url}?f=pjson", timeout=45)
print("Layer metadata status:", meta_resp.status_code)
meta_resp.raise_for_status()
meta = meta_resp.json()

meta_summary = {
    "name": meta.get("name"),
    "type": meta.get("type"),
    "geometryType": meta.get("geometryType"),
    "objectIdField": meta.get("objectIdField"),
    "displayField": meta.get("displayField"),
    "maxRecordCount": meta.get("maxRecordCount"),
}
print(meta_summary)

fields_df = pd.DataFrame(meta.get("fields", []))
print("Field count:", len(fields_df))
if not fields_df.empty:
    cols = [c for c in ["name", "alias", "type", "length", "nullable"] if c in fields_df.columns]
    display(fields_df[cols].sort_values("name").reset_index(drop=True))

query_url = f"{layer_url}/query"
params = {
    "f": "json",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 10,
}
sample_resp = requests.get(query_url, params=params, timeout=45)
print("Sample query status:", sample_resp.status_code)
sample_resp.raise_for_status()
sample_json = sample_resp.json()

rows = [f.get("attributes", {}) for f in sample_json.get("features", [])]
sample_df = pd.DataFrame(rows)
print("Sample row count:", len(sample_df))
if not sample_df.empty:
    preferred_cols = [c for c in ["SA", "Tyyppi", "Kulutus_25", "Tuotanto_1", "Tuotanto_2", "X", "Y"] if c in sample_df.columns]
    display(sample_df[preferred_cols] if preferred_cols else sample_df.head(10))